# Speed Dating - Full EDA & Storytelling Marketing

Source unique: `outputs/data/df_speed_dating_cleaned.csv`

Objectifs:
- faire une EDA lisible pour non-tech,
- repondre explicitement aux questions du brief,
- ajouter des insights marketing actionnables,
- produire des visualisations exportees en PNG (`outputs/images`).

In [11]:
# ============================================================================
# SETUP
# ============================================================================
from pathlib import Path
import shutil
import subprocess
import tempfile

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

PROJECT_ROOT = Path("../")
INPUT_PATH = PROJECT_ROOT / "outputs" / "data" / "df_speed_dating_cleaned.csv"
OUT_IMG = PROJECT_ROOT / "outputs" / "images"
OUT_IMG.mkdir(parents=True, exist_ok=True)

df = pd.read_csv(INPUT_PATH)

if "gender_label" not in df.columns and "gender" in df.columns:
    df["gender_label"] = df["gender"].map({0: "Femme", 1: "Homme"})
if "match_label" not in df.columns and "match" in df.columns:
    df["match_label"] = df["match"].map({0: "Pas de match", 1: "Match"})
if "gender" in df.columns:
    df["segment_genre"] = df["gender"].map({1: "Homme -> Femme", 0: "Femme -> Homme"})

ATTR_MAP = {
    "attr": "Attractivite", "sinc": "Sincerite", "intel": "Intelligence",
    "fun": "Fun", "amb": "Ambition", "shar": "Interets communs"
}


def export_figure(fig, name_no_ext: str, width: int = 4200, height: int = 2400, scale: int = 2):
    """Export PNG: Plotly/Kaleido puis fallback Chromium headless."""
    png_path = OUT_IMG / f"{name_no_ext}.png"

    try:
        fig.write_image(str(png_path), width=width, height=height, scale=scale)
        print(f"Export image (kaleido): {png_path}")
        return
    except Exception as kaleido_err:
        print(f"Kaleido indisponible ({type(kaleido_err).__name__}), fallback Chromium...")

    chromium_bin = shutil.which("chromium") or shutil.which("chromium-browser") or shutil.which("google-chrome")
    if not chromium_bin:
        raise RuntimeError("Export PNG impossible: ni Kaleido fonctionnel ni Chromium detecte.") from kaleido_err

    with tempfile.NamedTemporaryFile(suffix=".html", delete=False) as tmp:
        tmp_html = Path(tmp.name)

    try:
        fig.write_html(str(tmp_html), include_plotlyjs="cdn")
        cmd = [
            chromium_bin, "--headless", "--disable-gpu", "--no-sandbox",
            f"--window-size={width},{height}", f"--screenshot={png_path}", tmp_html.resolve().as_uri()
        ]
        subprocess.run(cmd, check=True, capture_output=True, text=True)
        print(f"Export image (chromium): {png_path}")
    finally:
        try:
            tmp_html.unlink(missing_ok=True)
        except Exception:
            pass

print(f"Dataset: {df.shape[0]:,} lignes x {df.shape[1]} colonnes")
print(f"Source: {INPUT_PATH}")

Dataset: 8,283 lignes x 50 colonnes
Source: ../outputs/data/df_speed_dating_cleaned.csv


## 1) Descriptive statistics (base du deliverable)

Cette section donne les statistiques descriptives essentielles pour contextualiser les analyses de second date.

In [12]:
# Stats descriptives
summary_cols = [c for c in ["match", "dec", "attr", "fun", "like", "prob", "int_corr", "samerace", "age", "go_out"] if c in df.columns]
print(df[summary_cols].describe(include="all").T)

match_rate = float(df["match"].mean() * 100)
dec_rate = float(df["dec"].mean() * 100) if "dec" in df.columns else np.nan
print(f"\nTaux de match global: {match_rate:.1f}%")
print(f"Taux de 'Oui' (dec=1): {dec_rate:.1f}%")

if "gender_label" in df.columns:
    print("\nTaux de match par genre:")
    print((df.groupby("gender_label")["match"].mean() * 100).round(2))

           count       mean       std    min    25%    50%    75%    max
match     8283.0   0.164433  0.370691   0.00   0.00   0.00   0.00   1.00
dec       8283.0   0.421224  0.493785   0.00   0.00   0.00   1.00   1.00
attr      8283.0   6.193094  1.931032   0.00   5.00   6.00   8.00  10.00
fun       8283.0   6.412411  1.913320   0.00   5.00   7.00   8.00  10.00
like      8283.0   6.130774  1.816015   0.00   5.00   6.00   7.00  10.00
prob      8283.0   5.207835  2.093503   0.00   4.00   5.00   7.00  10.00
int_corr  8283.0   0.195180  0.302263  -0.83  -0.01   0.21   0.43   0.91
samerace  8283.0   0.398165  0.489549   0.00   0.00   0.00   1.00   1.00
age       8283.0  26.358928  3.566763  18.00  24.00  26.00  28.00  55.00
go_out    8283.0   2.157189  1.105918   1.00   1.00   2.00   3.00   7.00

Taux de match global: 16.4%
Taux de 'Oui' (dec=1): 42.1%

Taux de match par genre:
gender_label
Femme    16.51
Homme    16.38
Name: match, dtype: float64


## 2) Reponses aux Data Exploration Ideas (avec graphes)

Ci-dessous, chaque question est traitee explicitement avec un graphe simple et interpretable.

In [13]:
# Q1) Least desirable attributes in partner, par genre
pairs = [
    ("attr1_1", "attr", "Attractivite"),
    ("sinc1_1", "sinc", "Sincerite"),
    ("intel1_1", "intel", "Intelligence"),
    ("fun1_1", "fun", "Fun"),
    ("amb1_1", "amb", "Ambition"),
    ("shar1_1", "shar", "Interets communs"),
]

pref_cols = [p for p, _, _ in pairs if p in df.columns]
pref_gender = df.groupby("gender_label", dropna=False)[pref_cols].mean().T.reset_index().rename(columns={"index": "pref_col"})
pref_gender["Attribut"] = pref_gender["pref_col"].map(dict((p, l) for p, _, l in pairs))
long_q1 = pref_gender.melt(id_vars=["pref_col", "Attribut"], var_name="Genre", value_name="Importance")

fig_q1 = px.bar(
    long_q1,
    x="Attribut", y="Importance", color="Genre", barmode="group",
    title="Q1 - Attributs les moins desires selon le genre",
    color_discrete_map={"Femme": "#D4537E", "Homme": "#378ADD"}
)
fig_q1.update_layout(height=700)
fig_q1.show()
export_figure(fig_q1, "story_q1_least_desirable")

print("Interpretation Q1:")
print("- Les barres les plus basses representent les attributs les moins prioritaires dans le declaratif.")
print("- Les differences Femme/Homme indiquent si le 'least desirable' change selon le genre.")

Export image (kaleido): ../outputs/images/story_q1_least_desirable.png
Interpretation Q1:
- Les barres les plus basses representent les attributs les moins prioritaires dans le declaratif.
- Les differences Femme/Homme indiquent si le 'least desirable' change selon le genre.


In [14]:
# Q2) Attractivite declaree vs impact reel sur le match
q2_df = pd.DataFrame({
    "Mesure": ["Importance declaree (attr1_1)", "Impact reel (corr attr vs match)"],
    "Valeur": [
        float(df["attr1_1"].mean()) if "attr1_1" in df.columns else np.nan,
        float(df["attr"].corr(df["match"])) if {"attr", "match"}.issubset(df.columns) else np.nan,
    ]
})
fig_q2 = px.bar(q2_df, x="Mesure", y="Valeur", color="Mesure", title="Q2 - Attractivite: declaree vs reelle")
fig_q2.update_layout(showlegend=False, height=700)
fig_q2.show()
export_figure(fig_q2, "story_q2_attr_declared_vs_real")

print("Interpretation Q2:")
print("- L'attractivite est souvent tres citee dans le declaratif.")
print("- Son impact reel est lu via la corrélation avec le match (niveau de prediction effectif).")

Export image (kaleido): ../outputs/images/story_q2_attr_declared_vs_real.png
Interpretation Q2:
- L'attractivite est souvent tres citee dans le declaratif.
- Son impact reel est lu via la corrélation avec le match (niveau de prediction effectif).


In [15]:
# Q3) Shared interests vs shared racial background
q3 = pd.DataFrame({
    "Variable": ["Interets communs (int_corr)", "Meme race (samerace)"],
    "Correlation avec match": [
        float(df["int_corr"].corr(df["match"])) if {"int_corr", "match"}.issubset(df.columns) else np.nan,
        float(df["samerace"].corr(df["match"])) if {"samerace", "match"}.issubset(df.columns) else np.nan,
    ]
})
fig_q3 = px.bar(q3, x="Variable", y="Correlation avec match", color="Variable", title="Q3 - Interets communs vs meme race")
fig_q3.update_layout(showlegend=False, height=700)
fig_q3.show()
export_figure(fig_q3, "story_q3_interests_vs_race")

print("Interpretation Q3:")
print("- La variable avec correlation absolue la plus forte est la plus informative pour predire un second date.")

Export image (kaleido): ../outputs/images/story_q3_interests_vs_race.png
Interpretation Q3:
- La variable avec correlation absolue la plus forte est la plus informative pour predire un second date.


In [16]:
# Q4) Les participants predisent-ils bien leur valeur percue?
if {"prob", "match"}.issubset(df.columns):
    calib = df[["prob", "match"]].dropna().copy()
    calib["prob_bin"] = pd.cut(calib["prob"], bins=[0,2,4,6,8,10], include_lowest=True)
    calib_grp = calib.groupby("prob_bin", observed=False).agg(
        prob_mean=("prob", "mean"),
        match_rate=("match", "mean"),
        n=("match", "size"),
    ).reset_index()
    calib_grp["match_rate_pct"] = (calib_grp["match_rate"] * 100).round(1)

    fig_q4 = px.line(
        calib_grp,
        x="prob_mean", y="match_rate_pct", markers=True,
        title="Q4 - Calibration: probabilite percue vs match reel",
        labels={"prob_mean": "Probabilite percue moyenne", "match_rate_pct": "Taux de match reel (%)"}
    )
    fig_q4.add_trace(go.Scatter(x=[0,10], y=[0,100], mode="lines", name="Calibration parfaite", line=dict(dash="dash", color="#888780")))
    fig_q4.update_layout(height=700)
    fig_q4.show()
    export_figure(fig_q4, "story_q4_calibration_perceived_value")

    print("Interpretation Q4:")
    print("- Plus la courbe est proche de la diagonale, meilleure est l'auto-evaluation du marche relationnel.")

Export image (kaleido): ../outputs/images/story_q4_calibration_perceived_value.png
Interpretation Q4:
- Plus la courbe est proche de la diagonale, meilleure est l'auto-evaluation du marche relationnel.


In [17]:
# Q5) First date vs last date
order_cols = [c for c in df.columns if any(k in c.lower() for k in ["order", "position", "first", "last", "date_no", "round"])]
print("Colonnes d'ordre detectees:", order_cols)

if len(order_cols) == 0:
    print("Conclusion Q5: non testable avec ce dataset nettoye (aucune variable d'ordre de passage).")
else:
    print("Une variable d'ordre existe: ajouter une analyse match rate par ordre.")

Colonnes d'ordre detectees: []
Conclusion Q5: non testable avec ce dataset nettoye (aucune variable d'ordre de passage).


## 3) Insights marketing complementaires (hors brief strict)

Cette partie ajoute des angles utiles au marketing pour actionner acquisition, onboarding et conversion.

In [18]:
# Superpredicteur like: taux de match selon seuil minimum
if {"like", "match"}.issubset(df.columns):
    thresholds = [4, 5, 6, 7, 8, 9]
    rates = [df[df["like"] >= t]["match"].mean() * 100 for t in thresholds]
    like_curve = pd.DataFrame({"Seuil like": thresholds, "Taux match (%)": rates})

    fig_like = px.line(like_curve, x="Seuil like", y="Taux match (%)", markers=True, title="Levier marketing: plus le like est haut, plus le match augmente")
    fig_like.update_layout(height=700)
    fig_like.show()
    export_figure(fig_like, "story_like_threshold_curve")

# Segmentation genre
if {"segment_genre", "match"}.issubset(df.columns):
    seg = (df.groupby("segment_genre")["match"].mean() * 100).reset_index(name="Taux match (%)")
    fig_seg = px.bar(seg, x="segment_genre", y="Taux match (%)", color="segment_genre", title="Segmentation genre et conversion")
    fig_seg.update_layout(showlegend=False, height=700)
    fig_seg.show()
    export_figure(fig_seg, "story_segmentation_genre")

Export image (kaleido): ../outputs/images/story_like_threshold_curve.png


Export image (kaleido): ../outputs/images/story_segmentation_genre.png


In [19]:
# Dashboard final storytelling (2x2)
match_rate = float(df["match"].mean() * 100)

corr_cols = [c for c in ["match", "like", "attr", "fun", "prob", "int_corr", "samerace"] if c in df.columns]
corr = df[corr_cols].corr(numeric_only=True)["match"].drop("match").sort_values(key=lambda s: s.abs(), ascending=False)

fig_dash = make_subplots(
    rows=2, cols=2,
    specs=[[{"type": "domain"}, {"type": "xy"}], [{"type": "xy"}, {"type": "xy"}]],
    subplot_titles=("Match rate global", "Top corrélations avec match", "Match vs attractivité", "Segmentation genre")
)

# 1 Donut
counts = df["match_label"].value_counts().reset_index()
counts.columns = ["Label", "Count"]
fig_dash.add_trace(go.Pie(labels=counts["Label"], values=counts["Count"], hole=0.55, textinfo="percent+label", showlegend=False), row=1, col=1)

# 2 Corr
fig_dash.add_trace(go.Bar(x=corr.values[::-1], y=[ATTR_MAP.get(c, c) for c in corr.index[::-1]], orientation="h", marker_color="#378ADD", showlegend=False), row=1, col=2)

# 3 Match vs attr bins
if {"attr", "match"}.issubset(df.columns):
    temp = df[["attr", "match"]].dropna().copy()
    temp["attr_bin"] = pd.cut(temp["attr"], bins=[0,2,4,6,8,10], labels=["0-2","2-4","4-6","6-8","8-10"], include_lowest=True)
    mattr = (temp.groupby("attr_bin", observed=False)["match"].mean() * 100).reset_index(name="Taux match (%)")
    fig_dash.add_trace(go.Bar(x=mattr["attr_bin"].astype(str), y=mattr["Taux match (%)"], marker_color="#1D9E75", showlegend=False), row=2, col=1)

# 4 Seg
if {"segment_genre", "match"}.issubset(df.columns):
    seg = (df.groupby("segment_genre")["match"].mean() * 100).reset_index(name="Taux match (%)")
    fig_dash.add_trace(go.Bar(x=seg["segment_genre"], y=seg["Taux match (%)"], marker_color=["#378ADD", "#D4537E"], showlegend=False), row=2, col=2)

fig_dash.update_layout(
    width=1500,
    height=800,
    title=f"Dashboard final storytelling - second date (match global: {match_rate:.1f}%)",
    title_x=0.5,
    font=dict(size=18),
    plot_bgcolor="white"
)
fig_dash.update_yaxes(title_text="Variables", row=1, col=2)
fig_dash.update_xaxes(title_text="Corrélation", row=1, col=2)
fig_dash.update_yaxes(title_text="Taux de match (%)", row=2, col=1)
fig_dash.update_yaxes(title_text="Taux de match (%)", row=2, col=2)

fig_dash.show()
export_figure(fig_dash, "story_dashboard_final")

Export image (kaleido): ../outputs/images/story_dashboard_final.png


## 4) Conclusion storytelling (pour marketing)

### Ce qui influence le plus le second date
- Les signaux comportementaux en session (`like`, `attr`, `fun`, `prob`) sont plus utiles que certaines declarations initiales.
- Le taux de match global reste faible: optimisation de la conversion necessaire a chaque etape du funnel.

### Recommandations actionnables
- Prioriser des scores d'intention dynamique (`like` et proxys) dans le ranking de relance.
- Adapter les messages par segment genre (`Homme -> Femme`, `Femme -> Homme`).
- Ne pas surpondérer les variables faibles (ex: `samerace` si correlation faible).

### Limite methodologique
- La question `first date vs last date` n'est pas testable ici faute de variable d'ordre dans le dataset nettoye.